## RAG Fusion Pipeline

This notebook demonstrates a complete RAG (Retrieval-Augmented Generation) pipeline built on top of our custom RAGFusion class. The pipeline uses LLM-generated sub-queries to improve retrieval quality, then fuses the results using Reciprocal Rank Fusion (RRF).

Steps covered:

1.Load the source PDF

2.Split documents into chunks

3.Generate embeddings and store in ChromaDB

4.Create a similarity-search retriever

5.Apply RAG Fusion (sub-query generation + RRF)

6.Augmentation - build context from retrieved documents

7.Generation - produce a grounded answer using an LLM

### Imports & Setup

In [4]:
from dotenv import load_dotenv
import sys
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate

sys.path.append(str(Path().resolve().parent))

from rag_fusion import RAGFusion

load_dotenv()

True

## Step 1 : Load the PDF

In [6]:
loader = PyPDFLoader("E:/Machine Learning and Data Science/Advanced-RAGs-Detailed/08 Rank Fusion\Documents/notebooklm_rag.pdf")
pages = loader.load()

print(f"Loaded {len(pages)} page(s) from the PDF.")

Loaded 3 page(s) from the PDF.


## Step 2 : Split Documents into Chunks

In [7]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)


chunks = splitter.split_documents(pages)

print(f"Split into {len(chunks)} chunk(s).")

Split into 19 chunk(s).


## Step 3 : Embeddings and Vector Store

In [8]:
embedding_model = GoogleGenerativeAIEmbeddings(model='gemini-embedding-001')

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="notebooklm_rag"
)

print("vector store create successfully")

vector store create successfully


## Step 4 : Create the Retriever

In [9]:
retriever = vectorstore.as_retriever(search_type='similarity',search_kwargs={"k": 3})

## step 5 : RAG Fusion

In [11]:
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')


# Build the RAG Fusion pipeline: LLM generates 2 sub-queries, retrieves docs for each, then fuses
rag_fusion = RAGFusion.from_llm(
    llm=llm,
    retriever=retriever,
    num_subqueries=2,
    k=3
)

In [12]:
query = "How does NotebookLM retrieve relevant information from uploaded documents?"

# This generates sub-queries, retrieves docs for each, and returns RRF-ranked results
fused_docs = rag_fusion.invoke(query)

print(f"Retrieved {len(fused_docs)} fused document(s).")
for i, doc in enumerate(fused_docs):
    print(f"\n--- Document {i + 1} ---")
    print(doc.page_content)

Retrieved 3 fused document(s).

--- Document 1 ---
search. Hybrid search combines the recall strengths of keyword matching with the semantic understanding of
dense retrieval.
Retrieved chunks are ranked and selected based on their relevance scores. To further improve retrieval
quality, NotebookLM may employ re-ranking techniques such as cross-encoder models, which score each
query-chunk pair more accurately than the initial bi-encoder retrieval step. The final set of retrieved chunks

--- Document 2 ---
vectors exist in the same semantic space, making cosine similarity a reliable measure of relevance.
The system then performs a nearest-neighbor search against the vector index to retrieve the top-k most
relevant document chunks. In practice, NotebookLM likely uses a combination of dense retrieval (vector
similarity) and sparse retrieval such as keyword-based BM25 matching, through a technique known as hybrid

--- Document 3 ---
rather than simple keyword matching. The vectors are stored

## Step 6 : Augmentation

In [13]:
# Join all retrieved chunks into one context block
context = "\n\n".join([doc.page_content for doc in fused_docs])

print(context)

search. Hybrid search combines the recall strengths of keyword matching with the semantic understanding of
dense retrieval.
Retrieved chunks are ranked and selected based on their relevance scores. To further improve retrieval
quality, NotebookLM may employ re-ranking techniques such as cross-encoder models, which score each
query-chunk pair more accurately than the initial bi-encoder retrieval step. The final set of retrieved chunks

vectors exist in the same semantic space, making cosine similarity a reliable measure of relevance.
The system then performs a nearest-neighbor search against the vector index to retrieve the top-k most
relevant document chunks. In practice, NotebookLM likely uses a combination of dense retrieval (vector
similarity) and sparse retrieval such as keyword-based BM25 matching, through a technique known as hybrid

rather than simple keyword matching. The vectors are stored in a vector index that supports efficient
nearest-neighbor search, enabling fast retriev

## Step 7 : Generation

In [14]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Use ONLY the context provided below to answer the question.
Be clear, concise, and accurate in your response.
If the answer is not present in the context, say "I don't know" - do not make up an answer.

Context:
{context}

Question: {question}

Answer:
""")

# Chain: prompt -> LLM
generation_chain = prompt | llm

response = generation_chain.invoke({"context": context, "question": query})

print(response.content)


NotebookLM retrieves relevant information through a hybrid search approach that combines keyword matching with semantic understanding. It converts user queries into embeddings and performs a nearest-neighbor search against a vector index to find the most relevant document chunks. The system may also use re-ranking techniques like cross-encoder models to improve retrieval quality.
